# 30 — Judge benchmarks (Kaggle, 2×T4)

Runs a jev-lite model on four judge benchmarks (`jevlite.data.public.judge_bench`): **ContextualJudgeBench**, **RAGBench**, **HaluBench**, **FaithBench**. Set up for **Stage B** (`stage_b-full-s42-r1`); NLI, Laya and Stage A already have predictions on the same records (`data/judge_bench/kaggle/preds`), so they are not re-run here.

jev-lite runs at three lengths:
- `stage_b_512`: Stage A's training length (comparable with `jevlite_512` of Stage A);
- `stage_b_1024`: Stage B's training length;
- `stage_b_8192`: ModernBERT's maximum, no truncation except the longest RAGBench documents (CUAD, TechQA).

Reports follow **eval v2** (`JUDGE_BENCH.md` section 9): a **held-out** slice (CJB, FaithBench, HaluBench, RAGBench ×5) and an **in-distribution** slice (the 7 RAGBench subsets Stage B trained on), each minus the passages seen in training (`docs/judge_bench/eval_v2/<slice>_exclude.txt`).

- **Settings:** Accelerator → GPU T4 x2, Internet → On, secret `HF_TOKEN` attached (the Stage B model is in the private repo `Nefelim584/jevlite-runs`).
- **GPUs:** `stage_b_8192` (the longest run, ~1 h) on GPU 0; `stage_b_512` → `stage_b_1024` on GPU 1 (~50 min).
- **Smoke run first:** run all cells with `LIMIT = 40` (40 records per benchmark, 20 pairs for CJB) and check speed in the logs. Then set `LIMIT = None` and **Save Version → Save & Run All (Commit)**.
- **Stage A instead:** `MODEL = "Nefelim584/jev-lite-stage-a"` (public repo, no token) and the queues of commit `9d33d64`.
- **Hand-off** (on the laptop): last cell.


In [ ]:
# Environment: code from GitHub at a fixed commit, kept outside /kaggle/working so it is not saved as output.
COMMIT = "Judge_bench"   # branch or commit hash; a hash makes the run reproducible

!rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
!cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
!pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
!pip install -q loguru
!python --version
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Parameters
LIMIT = 40                        # smoke run: records per benchmark (pairs/2 for CJB); None = full benchmarks
# a save_final folder: "hub:<owner>/<repo>/<folder>" (private, needs HF_TOKEN) or a whole public Hub repo
MODEL = "hub:Nefelim584/jevlite-runs/stage_b-full-s42-r1/final"
BENCHMARKS = ["contextual_judge_bench", "ragbench", "halubench", "faithbench"]
REPO = "/tmp/JEV_Judge"
WORK = "/tmp/judge_bench"         # converted data and the model: not saved
OUT = "/kaggle/working"           # notebook output: predictions, logs, reports

# One queue per GPU; the runs of a queue go one after another. name: predict_baseline.py arguments
QUEUES = {
    0: {"stage_b_8192": "--baseline jevlite --model {model} --max-len 8192 --fp16"},
    1: {"stage_b_512": "--baseline jevlite --model {model} --max-len 512 --fp16",
        "stage_b_1024": "--baseline jevlite --model {model} --max-len 1024 --fp16"},
}
# eval v2 slices: name → ids to leave out (the other slice + passages seen in training)
SLICES = {s: f"{REPO}/docs/judge_bench/eval_v2/{s}_exclude.txt" for s in ("heldout", "in_distribution")}


## 1. Model and data

The model folder has the `final/` layout (`encoder/`, `heads.pt`, `jevlite.yaml`). The converters are deterministic, so record ids match a local conversion and the eval v2 lists. Conversion takes a few minutes (~70 MB of downloads).


In [ ]:
import json, os, random, subprocess, sys
from pathlib import Path
from huggingface_hub import snapshot_download

if MODEL.startswith("hub:"):  # same as jevlite.model.resolve_checkpoint
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    owner, repo, sub = MODEL.removeprefix("hub:").split("/", 2)
    MODEL_DIR = Path(snapshot_download(f"{owner}/{repo}", allow_patterns=[f"{sub}/*"], token=os.environ.get("HF_TOKEN"),
                                       local_dir=f"{WORK}/model")) / sub
else:
    MODEL_DIR = Path(snapshot_download(MODEL, local_dir=f"{WORK}/model"))
print("model:", MODEL_DIR, sorted(os.listdir(MODEL_DIR)))
assert (MODEL_DIR / "heads.pt").exists(), "not a save_final folder (has the run finished and uploaded final/?)"

DATA = Path(WORK) / "data"
subprocess.run([sys.executable, f"{REPO}/scripts/convert_public.py", "--datasets", ",".join(BENCHMARKS),
                "--out", str(DATA), "--raw-dir", f"{WORK}/raw"], check=True, cwd=REPO)


In [ ]:
# Input files: the full benchmarks, or a fixed random sample of each for the smoke run.
def sample(name: str, n: int) -> Path:
    rows = [json.loads(line) for line in (DATA / f"{name}.jsonl").open()]
    rng = random.Random(0)
    if name == "contextual_judge_bench":  # whole pairs, both orders
        groups = set(rng.sample(sorted({r["group"] for r in rows}), n // 2))
        rows = [r for r in rows if r["group"] in groups]
    else:
        rows = rng.sample(rows, min(n, len(rows)))
    path = DATA / "smoke" / f"{name}.jsonl"
    path.parent.mkdir(exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
    return path

FILES = [str(sample(n, LIMIT)) if LIMIT else str(DATA / f"{n}.jsonl") for n in BENCHMARKS]
for f in FILES:
    print(f, sum(1 for _ in open(f)))

## 2. Predictions

Each GPU runs its queue in a background process; logs go to `OUT/logs/<run>.log`. `--resume` keeps finished predictions, so re-running this cell after an interruption continues where it stopped (in the same session). Records a model cannot encode (e.g. a prompt longer than its max_len) go to `<run>.jsonl.skipped.jsonl`.

In [ ]:
PREDS, LOGS = Path(OUT) / "preds", Path(OUT) / "logs"
PREDS.mkdir(exist_ok=True); LOGS.mkdir(exist_ok=True)

def queue_script(runs: dict) -> str:
    lines = ["set -x"]
    for name, args in runs.items():
        a = args.format(model=MODEL_DIR)
        lines.append(f"{sys.executable} {REPO}/scripts/predict_baseline.py {a} --data {' '.join(FILES)} "
                     f"--out {PREDS}/{name}.jsonl --resume > {LOGS}/{name}.log 2>&1")
    return "\n".join(lines)

procs = {}
for gpu, runs in QUEUES.items():
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs[gpu] = subprocess.Popen(["bash", "-c", queue_script(runs)], env=env, cwd=REPO)
    print(f"GPU {gpu}: {', '.join(runs)} (pid {procs[gpu].pid})")

In [ ]:
# Wait for both queues, printing the last progress line of every log each minute. As soon as a queue
# finishes, its models get eval v2 reports of their own (printed here, saved to OUT/report_gpu<N>/<slice>),
# so the GPU 1 results can be read in the log of a running commit before stage_b_8192 is done.
import time

def progress(name: str) -> str:
    log = LOGS / f"{name}.log"
    if not log.exists():
        return "waiting"
    text = log.read_text().replace("\r", "\n").strip().splitlines()
    return text[-1][:160] if text else "starting"

def report(names: list[str], out: str, n_boot: int) -> None:
    names = [n for n in names if (PREDS / f"{n}.jsonl").exists()]
    if not names:
        print("no predictions for", out); return
    for slice_name, exclude in SLICES.items():
        cmd = (f"{sys.executable} {REPO}/scripts/judge_bench_report.py --data {' '.join(FILES)} "
               f"--pred {' '.join(f'{n}={PREDS}/{n}.jsonl' for n in names)} "
               f"--exclude {exclude} --tokenizer {MODEL_DIR}/encoder "
               f"--out {OUT}/{out}/{slice_name} --bootstrap {n_boot}")
        print(cmd, flush=True)
        subprocess.run(cmd, shell=True, cwd=REPO)

N_BOOT = 0 if LIMIT else 1000
reported = set()
while True:
    for gpu, p in procs.items():
        if p.poll() is not None and gpu not in reported:
            reported.add(gpu)
            print(f"GPU {gpu} done (exit code {p.returncode}): {', '.join(QUEUES[gpu])}", flush=True)
            report(list(QUEUES[gpu]), f"report_gpu{gpu}", N_BOOT)
    if len(reported) == len(procs):
        break
    print(time.strftime("%H:%M"), " | ".join(f"{n}: {progress(n)}" for runs in QUEUES.values() for n in runs), flush=True)
    time.sleep(60)
for runs in QUEUES.values():
    for n in runs:
        print(f"== {n}:", progress(n))


## 3. Report

eval v2, per slice, on the records every run predicted (the 512 run skips a few dozen whose prompt alone is longer than 512 tokens, so they drop out for all). `fits512` = records that fit 512 tokens untruncated. The wait cell above already printed one report per GPU queue; this one compares all three runs. The comparison with Stage A, NLI and Laya is built on the laptop (hand-off below).


In [ ]:
# All runs side by side, per eval v2 slice (the per-queue reports are in OUT/report_gpu<N>).
report([n for runs in QUEUES.values() for n in runs], "report", N_BOOT)


## Hand-off (on the laptop, from the repo root)

```bash
kaggle kernels output <user>/<notebook-slug> -p data/judge_bench/stage_b   # preds/, logs/, report*/
```

Then Stage B next to Stage A, NLI and Laya on eval v2 (no GPU; Stage A's predictions are in `data/judge_bench/kaggle/preds`):

```bash
A=data/judge_bench/kaggle/preds B=data/judge_bench/stage_b/preds
for s in heldout in_distribution; do
  uv run python scripts/judge_bench_report.py --data data/public/{contextual_judge_bench,ragbench,halubench,faithbench}.jsonl \
    --pred stage_b_8192=$B/stage_b_8192.jsonl stage_b_1024=$B/stage_b_1024.jsonl stage_b_512=$B/stage_b_512.jsonl \
      stage_a_8192=$A/jevlite_8192.jsonl stage_a_512=$A/jevlite_512.jsonl nli=$A/nli.jsonl laya=$A/laya.jsonl \
    --exclude docs/judge_bench/eval_v2/${s}_exclude.txt --tokenizer data/runs/stage_a-full-s42/final/encoder \
    --out data/judge_bench/eval_v2_stage_b/$s --bootstrap 1000
done
```
